In [1]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv
import tiktoken

load_dotenv()

API_KEY = os.environ["GOOGLE_API_KEY"]

client = genai.Client(api_key=API_KEY)

### Why Tokens Matter: The Economics

Tokens are the Currency of AI. Here's what you need to know:
1. You pay per token (input + output)
- Every word you send osts money
- Every word the AI generates costs money

2. Different models have different prices

3. One word ≠ One token
- 'Explain' = 1 token
- 'database' 1 token... but uncommon words split into 2-3 tokens
- Punctuation, numbers, special characters all counts

4. Context Windows are limited
- Every prompt you send uses up your available 'memory'
- Long prompts = less space for convrsation
- Less space = worse context awareness = worse answers 

In [2]:
# Example 1: Short prompt
prompt_1 = 'Hi my name is anusha. What is your name?'

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_1

)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Hello Anusha! It's nice to meet you.

I don't have a name. I am a large language model, trained by Google."""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='joubaYmyL-TYvdIPlNSD-QU' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=32,
  prompt_token_count=13,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=13
    ),
  ],
  thoughts_token_count=459,
  total_token_count=504
) automatic_function_calling_history=[] parsed=None


In [3]:
print(f"Response text: {response.text}")
print(f"\n TOTAL TOKEN COUNT: {response.usage_metadata.total_token_count}")
print(f" Input tokens (prompt): {response.usage_metadata.prompt_token_count}")
print(f" Output tkens (response): {response.usage_metadata.candidates_token_count}")
print(f" Thought tokens (thinking): {response.usage_metadata.thoughts_token_count}")

Response text: Hello Anusha! It's nice to meet you.

I don't have a name. I am a large language model, trained by Google.

 TOTAL TOKEN COUNT: 504
 Input tokens (prompt): 13
 Output tkens (response): 32
 Thought tokens (thinking): 459


### Example 2: BAD PROMPT

Vague prompt leads to verbose, unfocused output

In [8]:
# Example 2: Bad prompt 
prompt_2 = "Explain database in brief"

# Get response
response_2 = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_2
)
print(f"Response text: {response_2.text}")
print(f" TOTAL TOKEN COUNT: {response_2.usage_metadata.total_token_count}")
print(f" Input tokens (prompt): {response_2.usage_metadata.prompt_token_count}")
print(f" Output tokens (response): {response_2.usage_metadata.candidates_token_count}")
print(f" Thought tokens (thinking): {response_2.usage_metadata.thoughts_token_count}")

Response text: A **database** is a structured collection of data, designed for efficient storage, management, and retrieval of information.

Think of it like a highly organized digital filing cabinet that allows you to quickly add, update, delete, and search for specific pieces of information.

It's essential for almost all modern applications and businesses, enabling them to store and access everything from customer details and product inventories to website user data.
 TOTAL TOKEN COUNT: 930
 Input tokens (prompt): 5
 Output tokens (response): 84
 Thought tokens (thinking): 841


### Example 3: Efficient prompt

In [10]:
# Example 3: Efficient prompt - Clear instrutions and constraints
prompt_3 = """You are a helpful assistant fo college students. Your job is to explain technical concepts in simple language. Use examples wherever needed. Avoid jargon and technical terms. Be friendly and encouraging. Keep responses concise but complete. 
Now explain what is a database to college student. Include a real-world example. Explain why daabases are important. Keep i under 5 sentences."""

# Get response
response_3 = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_3
)

print(f"Response text: {response_3.text}")
print(f" TOTAL TOKEN COUNT: {response_3.usage_metadata.total_token_count}")
print(f" Input tokens (prompt): {response_3.usage_metadata.prompt_token_count}")
print(f" Output tokens (response): {response_3.usage_metadata.candidates_token_count}")
print(f" Thought tokens (thinking): {response_3.usage_metadata.thoughts_token_count}")

Response text: Hey there!

Think of a database as a super organized digital filing cabinet for information. Just like your college library has a system to track every book – who borrowed it, when it's due – that's essentially a database at work. It helps keep all that info tidy and makes it super easy to find exactly what you're looking for, whenever you need it. This organization is crucial for everything from online shopping to your university's student records, making daily life smooth and efficient. It's basically the backbone for managing almost all digital information we interact with!
 TOTAL TOKEN COUNT: 534
 Input tokens (prompt): 76
 Output tokens (response): 117
 Thought tokens (thinking): 341


In [ ]:
# It's longer but the instructions are very clear so the model doesn't need to guess it generates fewer thought tokens and the total count is far less compared to previous example.